In [1]:
# Dataframe creation

import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

product_catalog = {
    "Laptop": {
        "base_price": 900,
        "base_cost": 650
    },
    "Monitor": {
        "base_price": 250,
        "base_cost": 160
    },
    "Keyboard": {
        "base_price": 80,
        "base_cost": 40
    },
    "Mouse": {
        "base_price": 40,
        "base_cost": 18
    },
    "Headphones": {
        "base_price": 150,
        "base_cost": 85
    }
}

quantity_ranges = {
    "Laptop": (1, 4),
    "Monitor": (1, 5),
    "Keyboard": (1, 8),
    "Mouse": (1, 10),
    "Headphones": (1, 6)
}

# Products generation
products = np.random.choice(
    list(product_catalog.keys()),
    n
)

# 2026 casual date generation
dates = pd.to_datetime(
    np.random.choice(
        pd.date_range(
            start="2026-01-01",
            end="2026-12-31"
        ),
        n
    )
)

# Region generation
regions = np.random.choice(
    ["North", "Center", "South"],
    n
)

# Price and cost variation
price_variation = np.random.uniform(
    0.90,
    1.10,
    n
)

cost_variation = np.random.uniform(
    0.95,
    1.05,
    n
)

# Quantity by product
units = [
    np.random.randint(
        quantity_ranges[p][0],
        quantity_ranges[p][1] + 1
    )
    for p in products
]

# Price by product
unit_prices = [
    product_catalog[p]["base_price"] * variation
    for p, variation in zip(
        products,
        price_variation
    )
]

# Cost by product
unit_costs = [
    product_catalog[p]["base_cost"] * variation
    for p, variation in zip(
        products,
        cost_variation
    )
]

df = pd.DataFrame({
    "date": dates,
    "product": products,
    "region": regions,
    "units": units,
    "unit_price": unit_prices,
    "unit_cost": unit_costs
})

# Sorting by date
df = df.sort_values(
    "date"
).reset_index(drop=True)

# Month extraction
df["month"] = (
    df["date"]
    .dt.to_period("M")
    .astype(str)
)

df.head(10)

,date,product,region,units,unit_price,unit_cost,month
0,2026-01-01,Monitor,Center,3,259.789874,164.494710,2026-01
1,2026-01-01,Mouse,Center,7,41.025905,17.873295,2026-01
2,2026-01-01,Laptop,Center,2,974.010291,668.510029,2026-01
3,2026-01-01,Headphones,South,4,148.503496,87.224245,2026-01
4,2026-01-02,Headphones,Center,3,159.694825,82.078412,2026-01
5,2026-01-02,Keyboard,North,4,86.405140,41.410340,2026-01
6,2026-01-02,Keyboard,Center,4,85.062573,38.865002,2026-01
7,2026-01-03,Mouse,North,4,43.540250,18.704373,2026-01
8,2026-01-03,Monitor,South,5,257.031780,155.742351,2026-01
9,2026-01-03,Mouse,South,8,37.218637,17.458643,2026-01


In [2]:
# Anomaly simulation
# Excessive discount on laptop in the South

mask_laptop_south = (
    (df["product"] == "Laptop") &
    (df["region"] == "South")
)

discounted_laptops = (
    df[mask_laptop_south]
    .sample(
        frac=0.20,
        random_state=42
    )
    .index
)

df.loc[
    discounted_laptops,
    "unit_price"
] *= 0.65

# Headphone cost increase in july

mask_headphones_cost = (
    (df["product"] == "Headphones") &
    (df["date"] >= "2026-07-01")
)

df.loc[
    mask_headphones_cost,
    "unit_cost"
] *= 1.30

df.head(10)

,date,product,region,units,unit_price,unit_cost,month
0,2026-01-01,Monitor,Center,3,259.789874,164.494710,2026-01
1,2026-01-01,Mouse,Center,7,41.025905,17.873295,2026-01
2,2026-01-01,Laptop,Center,2,974.010291,668.510029,2026-01
3,2026-01-01,Headphones,South,4,148.503496,87.224245,2026-01
4,2026-01-02,Headphones,Center,3,159.694825,82.078412,2026-01
5,2026-01-02,Keyboard,North,4,86.405140,41.410340,2026-01
6,2026-01-02,Keyboard,Center,4,85.062573,38.865002,2026-01
7,2026-01-03,Mouse,North,4,43.540250,18.704373,2026-01
8,2026-01-03,Monitor,South,5,257.031780,155.742351,2026-01
9,2026-01-03,Mouse,South,8,37.218637,17.458643,2026-01


In [3]:
# Economic metrics creation

df["revenue"] = df["units"] * df["unit_price"]
df["cost"] = df["units"] * df["unit_cost"]
df["profit"] = df["revenue"] - df["cost"]
df["margin"] = df["profit"] / df["revenue"]

df.head()

,date,product,region,units,unit_price,unit_cost,month,revenue,cost,profit,margin
0,2026-01-01,Monitor,Center,3,259.789874,164.494710,2026-01,779.369622,493.484129,285.885493,0.366816
1,2026-01-01,Mouse,Center,7,41.025905,17.873295,2026-01,287.181334,125.113068,162.068266,0.564341
2,2026-01-01,Laptop,Center,2,974.010291,668.510029,2026-01,1948.020583,1337.020059,611.000524,0.313652
3,2026-01-01,Headphones,South,4,148.503496,87.224245,2026-01,594.013983,348.896980,245.117003,0.412645
4,2026-01-02,Headphones,Center,3,159.694825,82.078412,2026-01,479.084475,246.235235,232.849240,0.486030


In [4]:
# Quality control

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1000 non-null   datetime64[ns]
 1   product     1000 non-null   object        
 2   region      1000 non-null   object        
 3   units       1000 non-null   int64         
 4   unit_price  1000 non-null   float64       
 5   unit_cost   1000 non-null   float64       
 6   month       1000 non-null   object        
 7   revenue     1000 non-null   float64       
 8   cost        1000 non-null   float64       
 9   profit      1000 non-null   float64       
 10  margin      1000 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(1), object(3)
memory usage: 86.1+ KB


In [5]:
df.isna().sum()

date          0
product       0
region        0
units         0
unit_price    0
unit_cost     0
month         0
revenue       0
cost          0
profit        0
margin        0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.describe()

,date,units,unit_price,unit_cost,revenue,cost,profit,margin
count,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2026-07-04 04:03:21.600000,3.813000,286.778412,197.652097,835.452503,562.100238,273.352265,0.400077
min,2026-01-01 00:00:00,1.000000,36.003542,17.105654,36.003542,17.261849,-364.481761,-0.220935
25%,2026-03-31 00:00:00,2.000000,75.651242,39.090349,261.632833,136.822591,110.371643,0.304564
50%,2026-07-05 12:00:00,3.000000,149.952102,105.165945,504.239373,306.480605,201.975417,0.404238
75%,2026-10-12 00:00:00,5.000000,265.494934,164.494888,954.325601,653.583439,332.989139,0.514184
max,2026-12-31 00:00:00,10.000000,989.096646,682.299099,3956.386586,2729.196397,1464.875736,0.608161
std,NaN,2.241664,317.662780,237.357210,894.336946,667.693104,257.152771,0.129374


In [8]:
# Impossible values control

print("Invalid units:", (df["units"] <= 0).sum())
print("Invalid prices:", (df["unit_price"] <= 0).sum())
print("Invalid costs:", (df["unit_cost"] < 0).sum())
print("Negative margins:", (df["margin"] < 0).sum())

Invalid units: 0
Invalid prices: 0
Invalid costs: 0
Negative margins: 14


In [9]:
# KPI

kpis = {
    "total_revenue": df["revenue"].sum(),
    "total_profit": df["profit"].sum(),
    "average_margin": df["profit"].sum() / df["revenue"].sum(),
    "total_units": df["units"].sum(),
    "average_order_value": df["revenue"].mean()
}

kpis

{'total_revenue': np.float64(835452.5032240103),
 'total_profit': np.float64(273352.2649378493),
 'average_margin': np.float64(0.32719067078377667),
 'total_units': np.int64(3813),
 'average_order_value': np.float64(835.4525032240103)}

In [10]:
# Products with best revenue

product_performance = (
    df.groupby("product")
      .agg(
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units=("units", "sum"),
          costs=("cost", "sum")
      )
)
product_performance["margin"] = (
    product_performance["profit"]
    / product_performance["revenue"]
)
product_performance = (
    product_performance
    .sort_values(
        "revenue",
        ascending=False
    )
)

product_performance

,revenue,profit,units,costs,margin
product,,,,,
Laptop,466772.008376,123794.717854,530,342977.290521,0.265215
Monitor,145949.630885,52857.483803,582,93092.147082,0.362163
Headphones,108517.019082,37220.592623,725,71296.426459,0.342993
Keyboard,69689.962481,34942.605133,868,34747.357348,0.501401
Mouse,44523.882400,24536.865524,1108,19987.016876,0.551094


In [11]:
# product_performance structuring

product_data = (
    product_performance
    .reset_index()
    .to_dict(orient="records")
)

display(product_data)

[{'product': 'Laptop',
  'revenue': 466772.0083758335,
  'profit': 123794.71785438621,
  'units': 530,
  'costs': 342977.2905214473,
  'margin': 0.2652145279343951},
 {'product': 'Monitor',
  'revenue': 145949.630885025,
  'profit': 52857.483802945666,
  'units': 582,
  'costs': 93092.14708207933,
  'margin': 0.36216250416272244},
 {'product': 'Headphones',
  'revenue': 108517.01908209173,
  'profit': 37220.59262278949,
  'units': 725,
  'costs': 71296.42645930224,
  'margin': 0.3429931354328171},
 {'product': 'Keyboard',
  'revenue': 69689.96248082479,
  'profit': 34942.60513330612,
  'units': 868,
  'costs': 34747.35734751867,
  'margin': 0.501400831474412},
 {'product': 'Mouse',
  'revenue': 44523.88240023534,
  'profit': 24536.865524421817,
  'units': 1108,
  'costs': 19987.016875813526,
  'margin': 0.5510944733851898}]

In [12]:
# Region with best revenue

region_performance = (
    df.groupby("region")
      .agg(
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units=("units", "sum"),
          costs=("cost", "sum")
      )
)
region_performance["margin"] = (
    region_performance["profit"]
    / region_performance["revenue"]
)
region_performance = (
    region_performance
    .sort_values(
        "revenue",
        ascending=False
    )
)
region_performance

,revenue,profit,units,costs,margin
region,,,,,
Center,295250.798970,98738.869642,1320,196511.929327,0.334424
South,276475.975849,84860.383286,1257,191615.592563,0.306936
North,263725.728405,89753.012010,1236,173972.716395,0.340327


In [13]:
# region_performance structuring

region_data = (
    region_performance
    .reset_index()
    .to_dict(orient="records")
)

display(region_data)

[{'region': 'Center',
  'revenue': 295250.798969503,
  'profit': 98738.86964207405,
  'units': 1320,
  'costs': 196511.92932742895,
  'margin': 0.33442371701176316},
 {'region': 'South',
  'revenue': 276475.9758490581,
  'profit': 84860.38328573856,
  'units': 1257,
  'costs': 191615.59256331951,
  'margin': 0.30693583059118323},
 {'region': 'North',
  'revenue': 263725.7284054493,
  'profit': 89753.01201003669,
  'units': 1236,
  'costs': 173972.7163954126,
  'margin': 0.3403270987351348}]

In [14]:
# Monthly performance

monthly_performance = (
    df.groupby("month")
      .agg(
          revenue=("revenue", "sum"),
          profit=("profit", "sum"),
          units=("units", "sum"),
          costs=("cost", "sum")
      )
)
monthly_performance["margin"] = (
    monthly_performance["profit"]
    / monthly_performance["revenue"]
)

# Trend
monthly_performance[
    "revenue_growth_pct"
] = (
    monthly_performance["revenue"]
    .pct_change()
    * 100
)
monthly_performance[
    "profit_growth_pct"
] = (
    monthly_performance["profit"]
    .pct_change()
    * 100
)

monthly_performance

,revenue,profit,units,costs,margin,revenue_growth_pct,profit_growth_pct
month,,,,,,,
2026-01,76633.205264,27513.292926,392,49119.912338,0.359026,NaN,NaN
2026-02,54088.218749,16859.842014,270,37228.376735,0.311710,-29.419344,-38.721105
2026-03,65806.648277,20980.425075,283,44826.223201,0.318819,21.665401,24.440223
2026-04,65286.239859,22112.353715,289,43173.886143,0.338699,-0.790814,5.395165
2026-05,63318.843293,21613.038260,230,41705.805033,0.341337,-3.013493,-2.258084
2026-06,74674.699416,25263.108312,329,49411.591104,0.338309,17.934402,16.888278
2026-07,54688.673772,18130.895779,298,36557.777993,0.331529,-26.764119,-28.231730
2026-08,80086.211614,25000.167698,341,55086.043916,0.312166,46.440215,37.887107
2026-09,60879.471141,19627.075639,295,41252.395502,0.322392,-23.982581,-21.492224


In [15]:
# monthly_performance structuring

month_data = (
    monthly_performance
    .reset_index()
    .to_dict(orient="records")
)

display(month_data)

[{'month': '2026-01',
  'revenue': 76633.20526415478,
  'profit': 27513.292925774742,
  'units': 392,
  'costs': 49119.91233838004,
  'margin': 0.359025736049228,
  'revenue_growth_pct': nan,
  'profit_growth_pct': nan},
 {'month': '2026-02',
  'revenue': 54088.21874906322,
  'profit': 16859.84201445973,
  'units': 270,
  'costs': 37228.37673460349,
  'margin': 0.3117100619023756,
  'revenue_growth_pct': -29.41934431344605,
  'profit_growth_pct': -38.721104522297104},
 {'month': '2026-03',
  'revenue': 65806.64827658796,
  'profit': 20980.425075329185,
  'units': 283,
  'costs': 44826.22320125878,
  'margin': 0.3188192321716132,
  'revenue_growth_pct': 21.665401077249747,
  'profit_growth_pct': 24.44022344536483},
 {'month': '2026-04',
  'revenue': 65286.23985854962,
  'profit': 22112.353715263922,
  'units': 289,
  'costs': 43173.886143285694,
  'margin': 0.33869853376719133,
  'revenue_growth_pct': -0.7908143503237652,
  'profit_growth_pct': 5.39516542620373},
 {'month': '2026-05',
 

In [16]:
# Anomaly detection

loss_making_sales = (
    df[
        df["profit"] < 0
    ]
)
print(
    "Loss-making transactions:",
    len(loss_making_sales)
)

loss_making_sales[
    [
        "date",
        "product",
        "region",
        "revenue",
        "cost",
        "profit",
        "margin"
    ]
].head(10)

Loss-making transactions: 14


,date,product,region,revenue,cost,profit,margin
47,2026-01-14,Laptop,South,599.975226,644.001525,-44.026299,-0.073380
79,2026-01-25,Laptop,South,2448.022837,2481.008992,-32.986155,-0.013475
99,2026-02-01,Laptop,South,1896.184978,1952.398568,-56.213590,-0.029646
166,2026-02-28,Laptop,South,2313.537331,2678.019092,-364.481761,-0.157543
201,2026-03-16,Laptop,South,1816.196492,1961.813063,-145.616571,-0.080177
246,2026-03-30,Laptop,South,1112.490144,1260.935691,-148.445547,-0.133435
388,2026-05-28,Laptop,South,622.882037,635.662032,-12.779996,-0.020518
412,2026-06-03,Laptop,South,1787.924671,2041.391548,-253.466877,-0.141766
618,2026-08-17,Laptop,South,597.344058,637.774274,-40.430216,-0.067683
717,2026-09-26,Laptop,South,2230.803992,2492.762653,-261.958661,-0.117428


In [17]:
# Major loss
# By revenue

worst_products = (
    loss_making_sales
    .groupby("product")
    .agg(
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        units=("units", "sum"),
        costs=("cost", "sum")
    )
    .sort_values(
        "profit",
        ascending=True
    )
)
display(worst_products)

# By margin
worst_product_by_margin = (
    product_performance["margin"]
    .idxmin()
)
worst_product_margin = (
    product_performance
    .loc[
        worst_product_by_margin,
        "margin"
    ]
)
print(worst_product_by_margin, worst_product_margin)

# By month 
worst_month = (
    monthly_performance[
        "revenue_growth_pct"
    ]
    .idxmin()
)
worst_month

,revenue,profit,units,costs
product,,,,
Laptop,20940.021469,-2316.429454,36,23256.450923


Laptop 0.2652145279343951


'2026-02'

In [18]:
# Output organizzation

analysis_output = {
    "metadata": {
        "rows": int(len(df)),
        "period_start": (
            df["date"]
            .min()
            .strftime("%Y-%m-%d")
        ),
        "period_end": (
            df["date"]
            .max()
            .strftime("%Y-%m-%d")
        ),
        "products": int(
            df["product"]
            .nunique()
        ),
        "regions": int(
            df["region"]
            .nunique()
        )
    },
    "kpis": {
        "total_revenue": round(
            float(
                kpis[
                    "total_revenue"
                ]
            ),
            2
        ),
        "total_profit": round(
            float(
                kpis[
                    "total_profit"
                ]
            ),
            2
        ),
        "average_margin": round(
            float(
                kpis[
                    "average_margin"
                ]
            ),
            4
        ),
        "total_units": int(
            kpis[
                "total_units"
            ]
        ),
        "average_order_value": round(
            float(
                kpis[
                    "average_order_value"
                ]
            ),
            2
        )
    },
    "product_performance": (
        product_data
    ),
    "region_performance": (
        region_data
    ),
    "monthly_performance": (
        month_data
    ),
    "best_product_by_revenue": (
        product_performance
        .index[0]
    ),
    "anomalies": {
        "loss_making_transactions": (
            int(
                (
                    df["profit"] < 0
                ).sum()
            )
        ),
        "worst_product_by_margin": {
            "product": (
                worst_product_by_margin
            ),
            "margin": round(
                float(
                    worst_product_margin
                ),
                4
            )
        },
        "worst_month_by_revenue_growth": (
            worst_month
        )
    }
}

In [19]:
# JSON output

import json

print(
    json.dumps(
        analysis_output,
        indent=4
    )
)

{
    "metadata": {
        "rows": 1000,
        "period_start": "2026-01-01",
        "period_end": "2026-12-31",
        "products": 5,
        "regions": 3
    },
    "kpis": {
        "total_revenue": 835452.5,
        "total_profit": 273352.26,
        "average_margin": 0.3272,
        "total_units": 3813,
        "average_order_value": 835.45
    },
    "product_performance": [
        {
            "product": "Laptop",
            "revenue": 466772.0083758335,
            "profit": 123794.71785438621,
            "units": 530,
            "costs": 342977.2905214473,
            "margin": 0.2652145279343951
        },
        {
            "product": "Monitor",
            "revenue": 145949.630885025,
            "profit": 52857.483802945666,
            "units": 582,
            "costs": 93092.14708207933,
            "margin": 0.36216250416272244
        },
        {
            "product": "Headphones",
            "revenue": 108517.01908209173,
            "profit": 37220.592

In [20]:
# Python Insight: Division in section and insight calculation

def generate_business_insights(analysis_output):

    insights = []

    kpis = analysis_output["kpis"]
    products = analysis_output["product_performance"]
    regions = analysis_output["region_performance"]
    months = analysis_output["monthly_performance"]
    anomalies = analysis_output["anomalies"]

    # 1. Loss-making transactions
    loss_count = anomalies["loss_making_transactions"]

    if loss_count > 0:
        insights.append({
            "severity": "high",
            "type": "profitability",
            "message": (
                f"{loss_count} transactions generated negative profit."
            )
        })

    # 2. Worst product by margin
    worst_product = anomalies[
        "worst_product_by_margin"
    ]

    insights.append({
        "severity": "medium",
        "type": "product_margin",
        "message": (
            f"{worst_product['product']} has the lowest margin "
            f"at {worst_product['margin']:.1%}."
        )
    })

    # 3. Best region
    best_region = max(
        regions,
        key=lambda x: x["revenue"]
    )

    insights.append({
        "severity": "info",
        "type": "regional_performance",
        "message": (
            f"{best_region['region']} generated the highest regional "
            f"revenue at €{best_region['revenue']:,.2f}."
        )
    })

    # 4. Worst month
    worst_month_name = anomalies[
        "worst_month_by_revenue_growth"
    ]

    worst_month_data = next(
        m for m in months
        if m["month"] == worst_month_name
    )

    worst_growth = worst_month_data[
        "revenue_growth_pct"
    ]

    if worst_growth < 0:
        severity = (
            "high"
            if worst_growth <= -20
            else "medium"
        )

        insights.append({
            "severity": severity,
            "type": "revenue_trend",
            "message": (
                f"Revenue declined by {abs(worst_growth):.1f}% "
                f"in {worst_month_name} compared with the previous month."
            )
        })

    # 5. Product margin vs portfolio
    portfolio_margin = kpis["average_margin"]

    low_margin_products = [
        p for p in products
        if p["margin"] < portfolio_margin
    ]

    if low_margin_products:

        weakest_margin_product = min(
            low_margin_products,
            key=lambda x: x["margin"]
        )

        margin_gap = (
            portfolio_margin
            - weakest_margin_product["margin"]
        )

        insights.append({
            "severity": "medium",
            "type": "relative_margin",
            "message": (
                f"{weakest_margin_product['product']} is "
                f"{margin_gap:.1%} below the portfolio average margin."
            )
        })

    return insights

# Insight printing
insights = generate_business_insights(
    analysis_output
)
for insight in insights:
    print(
        f"[{insight['severity'].upper()}] "
        f"{insight['message']}"
    )

[HIGH] 14 transactions generated negative profit.
[MEDIUM] Laptop has the lowest margin at 26.5%.
[INFO] Center generated the highest regional revenue at €295,250.80.
[HIGH] Revenue declined by 29.4% in 2026-02 compared with the previous month.
[MEDIUM] Laptop is 6.2% below the portfolio average margin.


In [21]:
# JSON update

analysis_output["insights"] = insights
print(
    json.dumps(
        analysis_output,
        indent=4
    )
)

{
    "metadata": {
        "rows": 1000,
        "period_start": "2026-01-01",
        "period_end": "2026-12-31",
        "products": 5,
        "regions": 3
    },
    "kpis": {
        "total_revenue": 835452.5,
        "total_profit": 273352.26,
        "average_margin": 0.3272,
        "total_units": 3813,
        "average_order_value": 835.45
    },
    "product_performance": [
        {
            "product": "Laptop",
            "revenue": 466772.0083758335,
            "profit": 123794.71785438621,
            "units": 530,
            "costs": 342977.2905214473,
            "margin": 0.2652145279343951
        },
        {
            "product": "Monitor",
            "revenue": 145949.630885025,
            "profit": 52857.483802945666,
            "units": 582,
            "costs": 93092.14708207933,
            "margin": 0.36216250416272244
        },
        {
            "product": "Headphones",
            "revenue": 108517.01908209173,
            "profit": 37220.592

In [22]:
# LLM usage

import json
from ollama import chat

analysis_json = json.dumps(
    analysis_output,
    indent=2
)

system_prompt = """
You are a senior business data analyst.

Your task is to analyze structured business performance data
and produce concise, evidence-based management insights.

STRICT RULES:

1. Use only facts and numerical values explicitly contained
   in the provided data.

2. Never invent, estimate, extrapolate, simulate or assume
   numerical values that are not explicitly provided.

3. Do not calculate new financial impacts, percentages,
   savings, ROI, targets or forecasts unless the required
   calculation can be performed exactly from the provided data.

4. If the data is insufficient to support a conclusion,
   explicitly state that further analysis is required.

5. Clearly distinguish:
   - FACT: directly supported by the data
   - INTERPRETATION: reasonable explanation that is not proven

6. Do not claim causality from correlation.
   For example, do not say that a margin difference is caused
   by logistics, pricing, sourcing or marketing unless the data
   explicitly proves it.

7. Focus on:
   - revenue
   - profit
   - margins
   - product performance
   - regional performance
   - monthly trends
   - detected anomalies

8. Recommendations must be based on observed evidence.
   Do not attach invented financial benefits or numerical targets
   to recommendations.

9. When uncertain, say so.

The goal is accuracy and reliability, not creativity.

CAUSALITY RULES:

- Never explain WHY a metric changed unless the provided data
  directly demonstrates the cause.

- A lower margin does NOT automatically imply higher logistics,
  pricing problems, sourcing inefficiency, marketing issues,
  customer acquisition costs, or operational inefficiency.

- A monthly decline does NOT prove seasonality or recurring behavior
  unless multiple years of data are available.

- When proposing a possible explanation, label it explicitly as:
  "Hypothesis to investigate: ..."

- Recommendations may suggest what should be investigated,
  but must not present an unverified explanation as fact.
"""

user_prompt = f"""
Analyze the following business performance data.

Return:

1. Executive summary
2. Key findings
3. Main risks
4. Opportunities
5. Recommended actions

BUSINESS DATA:

{analysis_json}
"""

response = chat(
    model="qwen3.5:4b",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    think = False,
    options={
        "num_ctx": 8192,
        "num_predict": 1500
    }
)
print(response.message.content)

### 1. Executive Summary
The business generated **€835,452.50** in total revenue and **€273,352.26** in profit over the period from January to December 2026, resulting in an average margin of **32.7%**. Despite a robust year-over-year financial outcome, there are significant operational inconsistencies: **14 transactions were loss-making**, and the primary revenue driver (Laptops) operates at a low margin (**26.5%**). Revenue volatility is pronounced, with extreme monthly swings (e.g., +46.4% in August vs. -29.4% in February), suggesting that trends observed so far cannot yet be classified as seasonal or recurring without multi-year data.

### 2. Key Findings
*   **Financial Overview:**
    *   **Total Revenue:** €835,452.50 across 3,813 units sold.
    *   **Total Profit:** €273,352.26.
    *   **Average Order Value (AOV):** €835.45.
*   **Product Performance:**
    *   **Top Revenue Generator:** Laptops generated **€466,772.01** (approx. 56% of total revenue).
    *   **Margin Leader

In [23]:
# JSON output structuring

from pydantic import BaseModel
from typing import Literal

class Finding(BaseModel):
    title: str
    evidence: str
    interpretation: str

class Risk(BaseModel):
    title: str
    evidence: str
    hypothesis_to_investigate: str | None = None

class Opportunity(BaseModel):
    title: str
    evidence: str
    action: str

class Recommendation(BaseModel):
    priority: Literal["high", "medium", "low"]
    action: str
    rationale: str

class BusinessReport(BaseModel):
    executive_summary: str
    key_findings: list[Finding]
    risks: list[Risk]
    opportunities: list[Opportunity]
    recommendations: list[Recommendation]

structured_system_prompt = """
You are a senior business data analyst.

Analyze the provided structured business data.

STRICT RULES:

- Use only facts and numerical values explicitly contained in the data.
- Never invent numerical values.
- Never claim causality unless the data directly proves it.
- Do not infer seasonality from a single year.
- If a cause is unknown, state that it is unknown.
- Possible explanations must be placed only in
  hypothesis_to_investigate.
- Evidence must contain only information supported by the data.
- Recommendations must follow logically from observed evidence.
- Accuracy is more important than creativity.
"""

structured_user_prompt = f"""
Analyze the following business performance data:

{analysis_json}
"""

response = chat(
    model="qwen3.5:4b",
    messages=[
        {
            "role": "system",
            "content": structured_system_prompt
        },
        {
            "role": "user",
            "content": structured_user_prompt
        }
    ],
    format=BusinessReport.model_json_schema(),
    think=False,
    options={
        "num_ctx": 8192,
        "num_predict": 1500,
        "temperature": 0
    }
)
report = BusinessReport.model_validate_json(
    response.message.content
)
print(
    report.model_dump_json(
        indent=4
    )
)

{
    "executive_summary": "The business generated €835,452.50 in total revenue with a profit of €273,352.26 (32.7% average margin) over the period 2026-01 to 2026-12. Revenue and unit sales were highest in Q4 (specifically October), while February recorded the lowest monthly performance with a significant revenue decline. The 'Laptop' product is the primary revenue driver but has the lowest margin (26.5%) and generated negative profit in 14 transactions.",
    "key_findings": [
        {
            "title": "Revenue Concentration",
            "evidence": "Total revenue of €835,452.50 was driven primarily by the 'Laptop' product (€466,772.01), representing 55.9% of total revenue.",
            "interpretation": "The business is highly dependent on Laptop sales for top-line growth."
        },
        {
            "title": "Margin Disparity",
            "evidence": "Keyboard (50.14%) and Mouse (55.11%) have significantly higher margins than Laptops (26.52%). The portfolio average ma